In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
import seaborn as sns


In [ ]:
base_path = r'D:\Dsoft\Projects\ML\data\road_accidents' + '\\'
sample_submission = base_path + r'sample_submission.csv'
train_path = base_path + r'train.csv'
test_path = base_path + r'test.csv'

In [ ]:
df = pd.read_csv(train_path)

In [ ]:
df

In [ ]:
cat_cols = ['road_type', \
        'lighting', \
        'weather', \
        'road_signs_present', \
        'public_road', \
        'time_of_day', \
        'holiday', \
        'school_season']

num_cols = [i for i in df.columns if i not in cat_cols + ['accident_risk', 'id']]

In [ ]:
num_cols

In [ ]:
sns.pairplot(df[num_cols], diag_kind='kde')
plt.suptitle("Pairwise Scatter Plots Between Numeric Variables", y=1.02)
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

# Flatten the axes array for easy iteration
axes = axes.flatten()

# Scatter plots for each column against index
for i, col in enumerate(num_cols):
    axes[i].scatter(df.index, df[col], color='C'+str(i))
    axes[i].set_title(f'Scatter Plot of {col}')
    axes[i].set_xlabel('Index')
    axes[i].set_ylabel(col)
    axes[i].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Separate categorical and numerical columns
categorical_cols = cat_cols
numerical_cols = [col for col in df.columns if col not in categorical_cols + ['id', 'accident_risk']]

# One-hot encode categorical columns
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_features = ohe.fit_transform(df[categorical_cols])
ohe_feature_names = ohe.get_feature_names_out(categorical_cols)
df_ohe = pd.DataFrame(ohe_features, columns=ohe_feature_names, index=df.index)

# Concatenate numerical columns and encoded categorical columns
df_encoded = pd.concat([df[numerical_cols + ['id', 'accident_risk']], df_ohe], axis=1)

df_encoded.head()

## Encoders OHE, LE

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import pandas as pd
import numpy as np

# Create copies for different encoding methods
df_label = df.copy()
df_onehot = df.copy()

# Label Encoding
label_encoders = {}
for col in cat_cols:
    label_encoders[col] = LabelEncoder()
    df_label[col] = label_encoders[col].fit_transform(df_label[col])

# One-Hot Encoding
onehot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
onehot_features = onehot_encoder.fit_transform(df_onehot[cat_cols])
onehot_feature_names = onehot_encoder.get_feature_names_out(cat_cols)
df_onehot_encoded = pd.DataFrame(onehot_features, columns=onehot_feature_names, index=df_onehot.index)
df_onehot = pd.concat([df_onehot.drop(columns=cat_cols), df_onehot_encoded], axis=1)

# Compare basic statistics of the target variable grouped by encoded features
print("Label Encoding Example (road_type):")
print(df_label.groupby('road_type')['accident_risk'].mean())

print("\nOne-Hot Encoding Example (first few columns):")
print(df_onehot[onehot_feature_names[:5]].head())

# Store encoders and encoded dataframes for later use
encoders = {
    'label': label_encoders,
    'onehot': onehot_encoder
}

encoded_dfs = {
    'label': df_label,
    'onehot': df_onehot
}

In [ ]:
def cat_to_num(df):
    cat_cols = ['road_type', \
            'lighting', \
            'weather', \
            'road_signs_present', \
            'public_road', \
            'time_of_day', \
            'holiday', \
            'school_season']
    for i in cat_cols:
        if i not in df.columns:
            print(f"Necessary columns do not exist: {i}")
            raise;
    
    # Boolean columns True:1, False:0
    bool_cols = ['road_signs_present', 'public_road' , 'holiday' , 'school_season']
    for i in bool_cols:
        df[i] = df[i].apply(lambda x: 1 if x else 0)
    
    # road_type ['urban' 'rural' 'highway']
    road_mapping = {'highway': 3, 'urban':2, 'rural': 1}
    df['road_type'] = df['road_type'].map(road_mapping)  

    # lighting ['daylight' 'dim' 'night']
    light_mapping = {'daylight': 1, 'dim':2, 'night': 3}
    df['lighting'] = df['lighting'].map(light_mapping)  

    # weather ['rainy' 'clear' 'foggy']
    weather = {'rainy': 3, 'clear':1, 'foggy': 2}
    df['weather'] = df['weather'].map(weather) 

    # time_of_day ['afternoon' 'evening' 'morning']
    time_of_day = {'afternoon': 2, 'evening':3, 'morning': 1}
    df['time_of_day'] = df['time_of_day'].map(time_of_day) 
    
    return df
    

In [ ]:
df.columns

In [ ]:
cat_cols = ['road_type', \
        'lighting', \
        'weather', \
        'road_signs_present', \
        'public_road', \
        'time_of_day', \
        'holiday', \
        'school_season']

In [ ]:
prep_df = cat_to_num(df)

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(prep_df[['road_type', 'num_lanes', 'curvature', 'speed_limit', 'lighting',
       'weather', 'road_signs_present', 'public_road', 'time_of_day',
       'holiday', 'school_season', 'num_reported_accidents', 'accident_risk']].corr(), annot=True, cmap='coolwarm')
plt.title('Categorical Features Correlation with Accident Severity')

In [ ]:
plt.figure(figsize=(24,18))
sns.heatmap(df_encoded.corr(), annot=True, cmap='coolwarm')
plt.title('Categorical Features Correlation with Accident Severity')

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import joblib

# columns to scale (keep id and target unchanged)
exclude_cols = ['id', 'accident_risk']
feature_cols = [c for c in prep_df.columns if c not in exclude_cols]

scaler = MinMaxScaler()
prep_norm = prep_df.copy()
prep_norm[feature_cols] = scaler.fit_transform(prep_df[feature_cols])

# quick sanity checks
print("Normalized dataframe shape:", prep_norm.shape)
print(prep_norm[feature_cols].agg(['min', 'max']).T)

# # persist scaler for later use
# joblib.dump(scaler, base_path + 'minmax_scaler.pkl')

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(prep_norm[['road_type', 'num_lanes', 'curvature', 'speed_limit', 'lighting',
       'weather', 'road_signs_present', 'public_road', 'time_of_day',
       'holiday', 'school_season', 'num_reported_accidents', 'accident_risk']], annot=True)
plt.title('Features Correlation with Accident Severity')

In [ ]:
    # Split prep_norm into train / validation / test sets.
    # If prep_norm isn't available, fall back to prep_df.
def split_data(df_encoded):    
    df_for_split = df_encoded.copy()

    # features (exclude id and target) and target
    feature_cols = [c for c in df_for_split.columns if c not in ('id', 'accident_risk')]
    X = df_for_split[feature_cols]
    y = df_for_split['accident_risk']

    # Use binned target for stratified splits (preserves distribution of continuous target)
    y_binned = pd.qcut(y, q=10, labels=False, duplicates='drop')

    # first split: hold out test set (15%)
    TEST_SIZE = 0.15
    RANDOM_STATE = 42
    X_train_val, X_test, y_train_val, y_test, ybv_train_val, ybv_test = train_test_split(
        X, y, y_binned, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_binned
    )

    # second split: from remaining, take validation set so that val is 15% of original
    VAL_FRACTION_OF_REMAINING = 0.15 / (1 - TEST_SIZE)  # 0.15 / 0.85
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val,
        test_size=VAL_FRACTION_OF_REMAINING,
        random_state=RANDOM_STATE,
        stratify=pd.qcut(y_train_val, q=10, labels=False, duplicates='drop')
    )

    # quick sanity checks
    print("Shapes:")
    print("  X_train:", X_train.shape, "y_train:", y_train.shape)
    print("  X_val:  ", X_val.shape,  "y_val:  ", y_val.shape)
    print("  X_test: ", X_test.shape, "y_test: ", y_test.shape)

    # Optionally inspect target distribution across splits
    print("\nTarget summary (median, min, max):")
    print("  train:", y_train.median(), y_train.min(), y_train.max())
    print("  val:  ", y_val.median(),   y_val.min(),   y_val.max())
    print("  test: ", y_test.median(),  y_test.min(),  y_test.max())
    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
encoded_splits = {}
for i in encoded_dfs:
    print("Encoder Type: ", i)
    X_train, X_val, X_test, y_train, y_val, y_test = split_data(encoded_dfs[i])
    encoded_splits[i] = {
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train, 
        "y_val": y_val, 
        "y_test": y_test
    }

In [ ]:
models = {}

In [ ]:
from sklearn.ensemble import RandomForestRegressor
randomforest  = {}
for i in encoded_splits:
    model = RandomForestRegressor(random_state=42)
    X_train, y_train = encoded_splits[i]["X_train"],encoded_splits[i]["y_train"]
    model.fit(X_train, y_train)
    randomforest[i] = model
models["RandomForest"] = randomforest


In [ ]:
from sklearn.linear_model import LinearRegression
linear_models = {}
for i in encoded_splits:
    linear_model = LinearRegression()
    X_train, y_train = encoded_splits[i]["X_train"],encoded_splits[i]["y_train"]
    linear_model.fit(X_train, y_train)
    linear_models[i] = linear_model
models["LinearModel"] = linear_models


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
gradient_model = {}
for i in encoded_splits:
    gb_model = GradientBoostingRegressor(random_state=42)
    X_train, y_train = encoded_splits[i]["X_train"], encoded_splits[i]["y_train"]
    gb_model.fit(X_train, y_train)
    gradient_model[i] = gb_model
models["GradientBoosting"] = gradient_model


In [ ]:
import xgboost as xgb
xgb_models = {}
for i in encoded_splits:
    xgb_model = xgb.XGBRegressor(random_state=42, n_jobs=-1, tree_method='hist')
    X_train, y_train = encoded_splits[i]["X_train"], encoded_splits[i]["y_train"]
    xgb_model.fit(X_train, y_train)
    xgb_models[i] = xgb_model
models["XGBoost"] = xgb_models

In [ ]:
models

In [ ]:
encoded_dfs

In [ ]:
for i in encoded_splits:
    print(f"With {i} encoder")

    X_test = encoded_splits[i]["X_test"]
    
    model = models["RandomForest"][i]
    y_pred = model.predict(X_test)

    print("RandomForestRegressor Results")
    print('RMSE:', mean_squared_error(y_test, y_pred))
    print('R2:', r2_score(y_test, y_pred))

    linear_model = models["LinearModel"][i]
    y_pred_linear = linear_model.predict(X_test)

    print("LinearRegression Results")
    print('RMSE (Linear):', mean_squared_error(y_test, y_pred_linear))
    print('R2 (Linear):', r2_score(y_test, y_pred_linear))

    gb_model = models["GradientBoosting"][i]
    y_pred_gb = gb_model.predict(X_test)

    print("GradientBoostingRegressor Results")
    print('RMSE (GB):', mean_squared_error(y_test, y_pred_gb))
    print('R2 (GB):', r2_score(y_test, y_pred_gb))

    xgb_model = models["XGBoost"][i]
    y_pred_xgb = xgb_model.predict(X_test)

    print("XGBoost Results")
    print('RMSE (XGBoost):', mean_squared_error(y_test, y_pred_xgb))
    print('R2 (XGBoost):', r2_score(y_test, y_pred_xgb))

In [ ]:
# Visualize XGBoost feature importances and permutation importance
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.inspection import permutation_importance

# Ensure feature names are available
try:
    feature_names = X_train.columns.tolist()
except Exception as e:
    raise RuntimeError("X_train is not defined. Run the training / split cells first.") from e

# 1) XGBoost built-in importance (gain/weight)
if hasattr(xgb_model, 'feature_importances_'):
    imp = xgb_model.feature_importances_
    fi = pd.Series(imp, index=feature_names).sort_values(ascending=False)
    top_n = min(30, len(fi))
    plt.figure(figsize=(8, max(4, top_n*0.25)))
    sns.barplot(x=fi.values[:top_n], y=fi.index[:top_n], palette='viridis')
    plt.title('XGBoost feature_importances_ (built-in)')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
    # save top features to csv
    try:
        fi.head(100).to_csv(base_path + 'xgb_feature_importances_builtin.csv')
    except Exception:
        pass
else:
    print('xgb_model has no feature_importances_ attribute')

# 2) Permutation importance on validation set (more reliable, model-agnostic)
try:
    if 'X_val' in globals() and 'y_val' in globals():
        print('Computing permutation importance on validation set (this may take a bit)...')
        perm = permutation_importance(xgb_model, X_val, y_val, n_repeats=10, random_state=42, n_jobs=-1)
        perm_series = pd.Series(perm.importances_mean, index=feature_names).sort_values(ascending=False)
        top_n2 = min(30, len(perm_series))
        plt.figure(figsize=(8, max(4, top_n2*0.25)))
        sns.barplot(x=perm_series.values[:top_n2], y=perm_series.index[:top_n2], palette='magma')
        plt.title('Permutation importance (validation set)')
        plt.xlabel('Mean decrease in score')
        plt.tight_layout()
        plt.show()
        # persist results
        try:
            perm_series.to_csv(base_path + 'xgb_permutation_importances_validation.csv')
        except Exception:
            pass
    else:
        print('X_val / y_val not found: skip permutation importance (run split_data first).')
except Exception as e:
    print('Permutation importance failed:', e)

# 3) Print top 10 features (both methods if available)
print('\nTop features (built-in XGBoost importance):')
try:
    print(fi.head(10).to_string())
except NameError:
    print('built-in importances not available')

print('\nTop features (permutation importance):')
try:
    print(perm_series.head(10).to_string())
except NameError:
    print('permutation importances not available')

# Optional: show a compact dataframe of top features side-by-side if both exist
try:
    both = pd.concat([fi, perm_series], axis=1)
    both.columns = ['xgb_builtin', 'permutation_mean']
    display(both.head(20))
except Exception:
    pass


In [ ]:
# 1. SHAP Analysis
import shap
import numpy as np
import matplotlib.pyplot as plt

print("Computing SHAP values (this may take a few minutes)...")

# Initialize the SHAP explainer with a workaround for base_score issue
# try:
#     # First try direct approach
#     explainer = shap.TreeExplainer(xgb_model)
# except ValueError as e:
print("Adjusting model parameters to handle base_score format issue...")
# Get the booster from the model
xgb_booster = xgb_model.get_params()

# Fix the base_score parameter
params = xgb_booster
print(params)
if 'base_score' in params:
    try:
        # Try to convert the string base_score to float
        base_score_str = params['base_score'].strip('[]')
        base_score = float(base_score_str.replace('E-', 'e-'))
        xgb_booster.set_param('base_score', str(base_score))
    except Exception as e:
        print(f"Could not convert base_score, using default: {e}")
        xgb_booster.set_param('base_score', '0.5')
    
    # Create explainer with modified booster
explainer = shap.TreeExplainer(xgb_booster)

# Use a sample of training data for summary plots (faster)
n_samples = min(1000, X_train.shape[0])
sample_idx = np.random.choice(X_train.shape[0], n_samples, replace=False)
X_sample = X_train.iloc[sample_idx]

# Calculate SHAP values
print("Calculating SHAP values...")
shap_values = explainer.shap_values(X_sample)

# Convert shap_values to numpy array if it's not already
if isinstance(shap_values, list):
    shap_values = np.array(shap_values)

# 1. Summary Plot (overall feature importance and impact)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, plot_size=(10, 8), show=False)
plt.title('SHAP Summary Plot')
plt.tight_layout()
plt.show()

# 2. Bar Plot (mean absolute SHAP values)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
plt.title('SHAP Feature Importance (mean |SHAP value|)')
plt.tight_layout()
plt.show()

# 3. Dependence plots for top 3 features
top_features = ['curvature', 'speed_limit', 'lighting_night']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, feature in enumerate(top_features):
    plt.sca(axes[i])
    shap.dependence_plot(feature, shap_values, X_sample, show=False)
    plt.title(f'SHAP Dependence Plot: {feature}')
plt.tight_layout()
plt.show()

# Save SHAP values and feature importance
shap_importance = pd.DataFrame({
    'feature': X_sample.columns,
    'mean_abs_shap': np.abs(shap_values).mean(0),
    'std_abs_shap': np.abs(shap_values).std(0)
}).sort_values('mean_abs_shap', ascending=False)

try:
    shap_importance.to_csv(base_path + 'shap_feature_importance.csv')
except Exception as e:
    print(f"Couldn't save SHAP importance: {e}")

# Display top 10 features by SHAP importance
print("\nTop 10 features by mean |SHAP value|:")
print(shap_importance.head(10).to_string())

In [ ]:
# 2. Partial Dependence Plots (PDP) and Individual Conditional Expectation (ICE)
from sklearn.inspection import partial_dependence
import matplotlib.pyplot as plt
import numpy as np

# Setup for PDP plots
top_features = ['curvature', 'speed_limit', 'lighting_night', 'weather_clear', 'num_reported_accidents']
n_cols = 3
n_rows = (len(top_features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
axes = axes.ravel()

# Compute and plot PDP for each feature
for idx, feature in enumerate(top_features):
    # Calculate partial dependence
    pdp_result = partial_dependence(
        xgb_model, X_train, [feature],
        kind='both',  # Calculate both PDP and ICE
        grid_resolution=50,
        n_jobs=-1
    )
    
    # Extract results
    feature_values = pdp_result['values'][0]
    pdp_mean = pdp_result['average'][0]
    ice_lines = pdp_result['individual'][0]
    
    # Plot
    ax = axes[idx]
    
    # Plot ICE lines (light)
    ice_samples = min(50, ice_lines.shape[0])  # Sample some ICE lines to avoid overcrowding
    ice_indices = np.random.choice(ice_lines.shape[0], ice_samples, replace=False)
    for ice_idx in ice_indices:
        ax.plot(feature_values, ice_lines[ice_idx], color='lightblue', alpha=0.1)
    
    # Plot PDP line (dark)
    ax.plot(feature_values, pdp_mean, color='darkblue', linewidth=2, label='PDP')
    
    # Customize plot
    ax.set_xlabel(feature)
    ax.set_ylabel('Accident Risk')
    ax.set_title(f'PDP + ICE Plot: {feature}')
    ax.grid(True, alpha=0.3)

# Remove empty subplots if any
for idx in range(len(top_features), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.show()

# Feature Interactions (2D PDP for top pairs)
top_pairs = [
    ('curvature', 'speed_limit'),
    ('lighting_night', 'weather_clear'),
    ('curvature', 'lighting_night')
]

fig, axes = plt.subplots(1, len(top_pairs), figsize=(20, 6))

for idx, (feature1, feature2) in enumerate(top_pairs):
    # Calculate 2D partial dependence
    pdp_result = partial_dependence(
        xgb_model, 
        X_train, 
        [feature1, feature2],
        grid_resolution=20,
        n_jobs=-1
    )
    
    # Create 2D PDP plot
    XX, YY = np.meshgrid(pdp_result['values'][0], pdp_result['values'][1])
    Z = pdp_result['average'][0].T
    
    # Plot
    im = axes[idx].contourf(XX, YY, Z, levels=20, cmap='viridis')
    axes[idx].set_xlabel(feature1)
    axes[idx].set_ylabel(feature2)
    axes[idx].set_title(f'2D PDP: {feature1} vs {feature2}')
    plt.colorbar(im, ax=axes[idx])

plt.tight_layout()
plt.show()

# Save PDP data for key features
pdp_data = {}
for feature in top_features:
    pdp_result = partial_dependence(xgb_model, X_train, [feature], grid_resolution=50)
    pdp_data[feature] = {
        'values': pdp_result['values'][0],
        'pdp': pdp_result['average'][0]
    }

try:
    import joblib
    joblib.dump(pdp_data, base_path + 'pdp_data.pkl')
except Exception as e:
    print(f"Couldn't save PDP data: {e}")

In [ ]:
# 3. Cross-validated Feature Importance
from sklearn.model_selection import KFold
from sklearn.inspection import permutation_importance
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Parameters
n_splits = 5
n_repeats = 30
cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Storage for results
cv_importance = []
feature_names = X_train.columns.tolist()

print(f"Computing {n_splits}-fold CV importance with {n_repeats} repeats per fold...")

# Compute importance for each fold
for fold, (train_idx, val_idx) in enumerate(cv.split(X_train)):
    print(f"Processing fold {fold + 1}/{n_splits}...")
    
    # Split data
    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_val = y_train.iloc[val_idx]
    
    # Train model
    fold_model = xgb.XGBRegressor(random_state=42, n_jobs=-1, tree_method='hist')
    fold_model.fit(X_fold_train, y_fold_train)
    
    # Compute permutation importance
    perm_importance = permutation_importance(
        fold_model, X_fold_val, y_fold_val,
        n_repeats=n_repeats,
        random_state=42,
        n_jobs=-1
    )
    
    # Store results
    cv_importance.append(pd.DataFrame({
        'feature': feature_names,
        'importance_mean': perm_importance.importances_mean,
        'importance_std': perm_importance.importances_std
    }))

# Combine results across folds
cv_results = pd.concat([df.set_index('feature') for df in cv_importance], axis=1)
cv_results.columns = [f"{col}_{i}" for i in range(n_splits) for col in ['mean', 'std']]

# Calculate aggregate statistics
final_importance = pd.DataFrame({
    'mean_importance': cv_results.filter(like='mean').mean(axis=1),
    'std_importance': cv_results.filter(like='mean').std(axis=1)
}).sort_values('mean_importance', ascending=False)

# Plot results with error bars
plt.figure(figsize=(12, 8))
top_n = 20
top_features = final_importance.head(top_n)

# Create error bar plot
plt.errorbar(
    x=top_features['mean_importance'],
    y=range(top_n),
    xerr=top_features['std_importance'],
    fmt='o',
    capsize=5,
    capthick=2,
    markersize=8,
    color='darkblue'
)

# Customize plot
plt.yticks(range(top_n), top_features.index)
plt.xlabel('Mean Importance (with std dev)')
plt.title(f'Cross-validated Feature Importance\n({n_splits} folds, {n_repeats} repeats per fold)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Save results
try:
    final_importance.to_csv(base_path + 'cv_feature_importance.csv')
except Exception as e:
    print(f"Couldn't save CV importance: {e}")

# Display top 10 features with confidence intervals
print("\nTop 10 features by cross-validated importance:")
print(final_importance.head(10).round(4).to_string())

In [ ]:
pip install catboost

### ChatGPT Code

In [ ]:
# ===============================
# 1️⃣ Imports
# ===============================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

# ===============================
# 2️⃣ Load Data
# ===============================
import os
print(os.curdir)
data = pd.read_csv(r"D:\Dsoft\Projects\ML\data\road_accidents\train.csv")  # replace with your file path

# Drop ID
data = data.drop(columns=['id'])

# ===============================
# 3️⃣ Feature Engineering
# ===============================

# Cyclic encoding for time_of_day
time_map = {'morning': 0, 'afternoon': 1, 'evening': 2}
data['time_hour'] = data['time_of_day'].map(time_map)
data['time_sin'] = np.sin(2 * np.pi * data['time_hour'] / 3)
data['time_cos'] = np.cos(2 * np.pi * data['time_hour'] / 3)
data = data.drop(columns=['time_of_day', 'time_hour'])

# Interaction features
data['speed_curvature'] = data['speed_limit'] * data['curvature']
data['lane_density'] = data['num_lanes'] / data['speed_limit']
data['accidents_per_lane'] = data['num_reported_accidents'] / (data['num_lanes'] + 1e-3)

# Log-transform for accident counts (reduces skew)
data['log_accidents'] = np.log1p(data['num_reported_accidents'])

In [ ]:

# ===============================
# 4️⃣ Define Features & Target
# ===============================
X = data.drop(columns=['accident_risk'])
y = data['accident_risk']

categorical_cols = ['road_type', 'lighting', 'weather']
boolean_cols = ['road_signs_present', 'public_road', 'holiday', 'school_season']
numeric_cols = [c for c in X.columns if c not in categorical_cols + boolean_cols]

# ===============================
# 5️⃣ Preprocessing
# ===============================
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('bool', 'passthrough', boolean_cols),
    ('num', 'passthrough', numeric_cols)
])

# ===============================
# 6️⃣ Model and Hyperparameter Tuning
# ===============================
xgb = XGBRegressor(objective='reg:squarederror', random_state=42)

param_dist = {
    'n_estimators': [300, 500, 800],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [4, 6, 8],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.9, 1.0],
    'reg_lambda': [0.5, 1.0, 1.5],
    'min_child_weight': [1, 3, 5]
}

pipe = Pipeline([
    ('preprocess', preprocessor),
    ('model', xgb)
])

search = RandomizedSearchCV(
    pipe, param_distributions={'model__' + k: v for k, v in param_dist.items()},
    n_iter=20, scoring='r2', cv=5, verbose=2, n_jobs=-1, random_state=42
)

# ===============================
# 7️⃣ Train / Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ===============================
# 8️⃣ Train and Evaluate
# ===============================
search.fit(X_train, y_train)

best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"✅ Best R²: {r2:.4f}")
print(f"✅ RMSE: {rmse:.4f}")
print("Best Parameters:", search.best_params_)


In [ ]:
X_train.head()

In [ ]:
# -------------------------------
# 1. Imports
# -------------------------------
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# -------------------------------
# 2. Load your data
# -------------------------------
# Replace with your CSV or dataframe
# data = pd.read_csv("your_data.csv")

# Sample data snippet for illustration
data = pd.read_csv(r"D:\Dsoft\Projects\ML\data\road_accidents\train.csv")

# -------------------------------
# 3. Preprocessing
# -------------------------------

# Convert categorical features to one-hot (except time_of_day)
cat_cols = ['road_type', 'lighting', 'weather']
data = pd.get_dummies(data, columns=cat_cols, drop_first=True)

# Convert boolean features to int
bool_cols = ['road_signs_present', 'public_road', 'holiday', 'school_season']
for col in bool_cols:
    data[col] = data[col].astype(int)

# Cyclic encoding for time_of_day
time_map = {'morning': 0, 'afternoon': 1, 'evening': 2}
data['time_idx'] = data['time_of_day'].map(time_map)
n_times = 3
data['time_sin'] = np.sin(2 * np.pi * data['time_idx'] / n_times)
data['time_cos'] = np.cos(2 * np.pi * data['time_idx'] / n_times)
data.drop(columns=['time_of_day','time_idx'], inplace=True)

# Features and target
X = data.drop(columns=['accident_risk'])
y = data['accident_risk']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------------
# 4. Define Base Models
# -------------------------------
xgb = XGBRegressor(
    n_estimators=800, learning_rate=0.01, max_depth=8, subsample=0.7,
    colsample_bytree=1.0, reg_lambda=0.5, min_child_weight=1, random_state=42,
    verbosity=0
)

lgbm = LGBMRegressor(
    n_estimators=800, learning_rate=0.01, max_depth=8,
    subsample=0.7, colsample_bytree=1.0, reg_lambda=0.5, random_state=42
)

cat = CatBoostRegressor(
    iterations=800, learning_rate=0.01, depth=8, subsample=0.7,
    random_seed=42, verbose=False
)

# -------------------------------
# 5. Stacking Regressor
# -------------------------------
stack = StackingRegressor(
    estimators=[('xgb', xgb), ('lgbm', lgbm), ('cat', cat)],
    final_estimator=LinearRegression(),
    passthrough=True,
    n_jobs=-1
)

# -------------------------------
# 6. Train and Evaluate
# -------------------------------
stack.fit(X_train, y_train)
y_pred = stack.predict(X_test)

# Evaluation
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Stacking Regressor RMSE: {rmse:.4f}")
print(f"Stacking Regressor R²: {r2:.4f}")


### Data Analysis and Visualization


In [ ]:
numerical_cols = ['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents', 'accident_risk']

data[numerical_cols].hist(bins=30, figsize=(12,8), color='skyblue', edgecolor='black')
plt.suptitle("Distribution of Numerical Features", fontsize=16)
plt.show()

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(data=data[numerical_cols])
plt.title("Boxplot of Numerical Features")
plt.show()


In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(data=data[numerical_cols])
plt.title("Boxplot of Numerical Features")
plt.show()


In [ ]:
import os
os.listdir()

In [ ]:
# Full Data Visualization Script

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set(style="whitegrid")

# Load data
data = pd.read_csv(r"../data/road_accidents/train.csv")  # Replace with your CSV path

# ===============================
# 1️⃣ Numerical Features
# ===============================
numerical_cols = ['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents', 'accident_risk']

# a) Histograms
data[numerical_cols].hist(bins=30, figsize=(12,8), color='skyblue', edgecolor='black')
plt.suptitle("Distribution of Numerical Features", fontsize=16)
plt.show()

# b) Boxplots
plt.figure(figsize=(12,6))
sns.boxplot(data=data[numerical_cols])
plt.title("Boxplot of Numerical Features")
plt.show()

# c) Pairplot
sns.pairplot(data[numerical_cols])
plt.suptitle("Pairplot of Numerical Features", y=1.02)
plt.show()

# d) Correlation heatmap
plt.figure(figsize=(10,8))
sns.heatmap(data[numerical_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

# ===============================
# 2️⃣ Categorical Features
# ===============================
categorical_cols = ['road_type', 'lighting', 'weather', 'time_of_day']

# a) Countplots
for col in categorical_cols:
    plt.figure(figsize=(6,4))
    sns.countplot(data=data, x=col, palette='Set2')
    plt.title(f"Count of {col}")
    plt.show()

# b) Boxplot of target by category
for col in categorical_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=col, y='accident_risk', data=data, palette='Set3')
    plt.title(f"Accident Risk by {col}")
    plt.show()

# ===============================
# 3️⃣ Boolean Features
# ===============================
bool_cols = ['road_signs_present', 'public_road', 'holiday', 'school_season']

for col in bool_cols:
    plt.figure(figsize=(6,4))
    sns.barplot(x=col, y='accident_risk', data=data, ci=None)
    plt.title(f"Accident Risk by {col}")
    plt.show()

# ===============================
# 4️⃣ Feature Interactions
# ===============================
# Curvature vs Accident Risk by Road Type
sns.scatterplot(data=data, x='curvature', y='accident_risk', hue='road_type', palette='Set1')
plt.title("Curvature vs Accident Risk by Road Type")
plt.show()

# Speed Limit vs Accident Risk by Time of Day
sns.scatterplot(data=data, x='speed_limit', y='accident_risk', hue='time_of_day', palette='Set2')
plt.title("Speed Limit vs Accident Risk by Time of Day")
plt.show()

# ===============================
# 5️⃣ Cyclic Feature Visualization (time_of_day)
# ===============================
# Convert time_of_day to numeric indices
time_map = {'morning': 0, 'afternoon': 1, 'evening': 2}
data['time_idx'] = data['time_of_day'].map(time_map)

# Sin encoding
plt.figure(figsize=(6,4))
sns.scatterplot(x=np.sin(2*np.pi*data['time_idx']/3), 
                y='accident_risk')
plt.xlabel('Sin(Time of Day)')
plt.ylabel('Accident Risk')
plt.title("Accident Risk by Time of Day (Sin Encoding)")
plt.show()


In [ ]:
pip install tensorflow

In [ ]:
import tensorflow as tf
print(tf.__version__)
from tensorflow.keras import layers as layers


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

# Load data
df = pd.read_csv(r"../data/road_accidents/train.csv")  # Replace with your CSV path
df = df.drop(columns=['id'])  # Drop ID column
target = "accident_risk"  # replace with actual column
X = pd.get_dummies(df.drop(columns=[target]), drop_first=True)
y = df[target].values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Build neural network
model = keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)  # Linear output for regression
])

# Compile
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss='mse', metrics=['mae'])

# Train
history = model.fit(X_train, y_train, 
                    validation_split=0.2, 
                    epochs=100, 
                    batch_size=64,
                    verbose=1)

# Evaluate
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f"Test MAE: {test_mae:.4f}, Test MSE: {test_loss:.4f}")


In [ ]:
dir(model)

In [ ]:
model.input_shape

In [ ]:
z = model.predict(X_test)

In [ ]:
pd.concat([pd.Series(y_test, name='Actual'), pd.Series(z.flatten(), name='Predicted')], axis=1)

In [ ]:
rmse = root_mean_squared_error(y_test, z)
r2 = r2_score(y_test, z)
r2, rmse

In [ ]:
import pandas as pd
import numpy as np

# --- Step 1: Load Data ---
df = pd.read_csv(r"../data/road_accidents/train.csv")

# --- Step 2: Basic Cleaning ---
# Ensure boolean and categorical fields are properly typed
bool_cols = ['road_signs_present', 'public_road', 'holiday', 'school_season']
for col in bool_cols:
    df[col] = df[col].astype(bool)

# --- Step 3: Domain-Informed Features ---

# 1️⃣ Lighting Conditions
df['is_dark'] = df['lighting'].isin(['dim', 'night']).astype(int)

# 2️⃣ Weather Risk Scoring
weather_map = {'clear': 0, 'rainy': 1, 'foggy': 2}
df['weather_risk'] = df['weather'].map(weather_map)

# 3️⃣ Curvature Risk Level
df['curvature_category'] = pd.cut(
    df['curvature'],
    bins=[0, 0.3, 0.7, 1],
    labels=['low', 'medium', 'high']
)

# 4️⃣ Combined Visibility Risk
df['visibility_risk'] = df['weather_risk'] * df['is_dark']

# 5️⃣ Rush Hour Indicator
df['is_rush_hour'] = df['time_of_day'].isin(['morning', 'evening']).astype(int)

# 6️⃣ Time of Day Cyclic Encoding (for models)
time_map = {'morning': 0, 'afternoon': 1, 'evening': 2}
df['sin_time'] = np.sin(2 * np.pi * df['time_of_day'].map(time_map) / 3)
df['cos_time'] = np.cos(2 * np.pi * df['time_of_day'].map(time_map) / 3)

# 7️⃣ Combined Speed-Curvature-Darkness Risk
df['risk_combo'] = df['speed_limit'] * df['curvature'] * df['is_dark']

# 8️⃣ Lane-Speed Interaction
df['lane_speed_factor'] = df['num_lanes'] * df['speed_limit']

# 9️⃣ Sign Effectiveness (signs on curved roads)
df['sign_effectiveness'] = df['road_signs_present'].astype(int) * (df['curvature'] > 0.5).astype(int)

# 🔟 Activity Intensity (holidays or school in season)
df['is_high_activity'] = (df['holiday'] | df['school_season']).astype(int)

# --- Step 4: Encode Categorical Columns (optional, for ML) ---
categorical_cols = ['road_type', 'lighting', 'weather', 'time_of_day', 'curvature_category']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# --- Step 5: Save Processed Data ---
df_encoded.to_csv("processed_accident_data.csv", index=False)

print("✅ Feature engineering complete!")
print(f"Original columns: {df.shape[1]}, After encoding: {df_encoded.shape[1]}")
print("\nSample of engineered features:")
print(df_encoded.head())


In [ ]:
from xgboost import XGBRegressor, XGBRFRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

bool_cols = [
    'road_signs_present',
    'public_road',
    'holiday',
    'school_season',
    'road_type_rural',
    'road_type_urban',
    'lighting_dim',
    'lighting_night',
    'weather_foggy',
    'weather_rainy',
    'time_of_day_evening',
    'time_of_day_morning',
    'curvature_category_medium',
    'curvature_category_high'
]

df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

X = df_encoded.drop(columns=['accident_risk'])
y = df_encoded['accident_risk']
X_z = X.copy()
X_z.drop(columns=['id'], inplace=True)  # drop id if exists

X_train, X_test, y_train, y_test = train_test_split(X_z, y, test_size=0.2, random_state=42)
xgb_model = XGBRegressor(
    n_estimators=500,       # number of trees
    learning_rate=0.5,      # step size shrinkage
    max_depth=4,            # depth of each tree
    subsample=1.0,          # fraction of samples used per tree
    colsample_bytree=0.8,   # fraction of features used per tree
    random_state=42,
    n_jobs=-1,
    tree_method='hist',      # efficient tree method
    objective='reg:squarederror'  # regression task
)
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)
print('RMSE:', mean_squared_error(y_test, y_pred))
print('R2 Score:', r2_score(y_test, y_pred))


xgbrf_model = XGBRFRegressor(
    n_estimators=500,       # number of trees
    max_depth=4,            # depth of each tree
    subsample=1.0,          # fraction of samples used per tree
    colsample_bytree=0.8,   # fraction of features used per tree
    random_state=42,
    n_jobs=-1,
    tree_method='hist',      # efficient tree method
    objective='reg:squarederror'  # regression task
)
y_pred_rf = xgbrf_model.fit(X_train, y_train).predict(X_test)
print('RMSE (XGBRF):', mean_squared_error(y_test, y_pred_rf))
print('R2 Score (XGBRF):', r2_score(y_test, y_pred_rf))

In [ ]:
from xgboost import plot_importance
plot_importance(xgb_model)

In [ ]:
xgb_model

In [ ]:
import time
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
import os

safe_n_jobs = max(1, os.cpu_count() - 1)
best_xgb = None
# use smaller candidate space and n_iter modest
xgb_pipe = Pipeline([('xgb', XGBRegressor(tree_method='hist', objective='reg:squarederror', random_state=42, n_jobs=safe_n_jobs))])
xgb_param_dist = {
    'xgb__n_estimators': [100,200,300,500],
    'xgb__max_depth': [3,5,7],
    'xgb__learning_rate': [0.01,0.05,0.1],
    'xgb__subsample': [0.6,0.8,1.0],
    'xgb__colsample_bytree': [0.6,0.8,1.0]
}
scoring_metrics = ['r2', 'neg_root_mean_squared_error', 'neg_mean_absolute_error']
for i in scoring_metrics:
    print("XGBoost RandomizedSearchCV optimizing for:", i)
    xgb_search = RandomizedSearchCV(xgb_pipe, xgb_param_dist, n_iter=20, cv=3, scoring=i, n_jobs=safe_n_jobs, random_state=42, verbose=2)
    print("Starting XGBoost RandomizedSearchCV...")
    t0 = time.time(); 
    xgb_search.fit(X_train, y_train); 
    dt_x = time.time()-t0
    print("XGB search elapsed:", dt_x, "best score:", xgb_search.best_score_, "best params:", xgb_search.best_params_)
    # best_xgb = xgb_search.best_estimator_

In [ ]:
y_train_preds = xgb_model.predict(X_train)

In [ ]:
from lightgbm import LGBMRegressor
lgbm_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
lgbm_model.fit(X_train, y_train)
y_pred_lgbm = lgbm_model.predict(X_test)

print('RMSE (LGBM):', mean_squared_error(y_test, y_pred_lgbm))
print('R2 Score (LGBM):', r2_score(y_test, y_pred_lgbm))


from matplotlib import pyplot as plt
import seaborn as sns
# 1) Built-in feature importance from XGBoost
feature_names = X_train.columns.tolist()
if hasattr(lgbm_model, 'feature_importances_'):
    importances = lgbm_model.feature_importances_
    fi = pd.Series(importances, index=feature_names).sort_values(ascending=False)
    top_n = min(30, len(fi))
    plt.figure(figsize=(8, max(4, top_n*0.25)))
    sns.barplot(x=fi.values[:top_n], y=fi.index[:top_n], palette='viridis')
    plt.title('LGBM Built-in Feature Importances')
    plt.xlabel('Importance Score')
    plt.tight_layout()
    plt.show()

if hasattr(xgb_model, 'feature_importances_'):
    importances = xgb_model.feature_importances_
    fi = pd.Series(importances, index=feature_names).sort_values(ascending=False)
    top_n = min(30, len(fi))
    plt.figure(figsize=(8, max(4, top_n*0.25)))
    sns.barplot(x=fi.values[:top_n], y=fi.index[:top_n], palette='viridis')
    plt.title('XGBoost Built-in Feature Importances')
    plt.xlabel('Importance Score')
    plt.tight_layout()
    plt.show()

# 🚗 Accident Risk Prediction — Model Results & Interpretation

Excellent — you’ve built a very strong accident-risk prediction pipeline using **feature engineering + XGBoost + LightGBM**.  

Let’s break down the **results and interpretation**:

---

## ⚙️ 1. Model Performance Summary

| Model                             | RMSE    | R²         | Comments                                                                     |
| --------------------------------- | ------- | ---------- | ---------------------------------------------------------------------------- |
| **LightGBM**                      | 0.00317 | **0.8851** | Best generalization, efficient training, good handling of categorical splits |
| **XGBoost**                       | 0.00319 | 0.8843     | Nearly identical to LGBM, slightly higher bias                               |
| **XGBRF (XGBoost Random Forest)** | 0.00564 | 0.7957     | Underfits — likely due to smaller learning flexibility                       |

✅ **Observation:**  
Both **LGBM** and **XGBoost** are performing extremely well (R² ≈ 0.885).  
**LightGBM** wins by a small margin, showing that your feature extraction is effective and the model is robust.

---

## 📊 2. Feature Importance Insights

| Rank | Feature                                    | Model Insights                                                                                  |
| ---- | ------------------------------------------ | ----------------------------------------------------------------------------------------------- |
| 1    | **Curvature**                              | Steep curves strongly increase accident risk — expected, as sharp turns reduce driver control.  |
| 2    | **Risk combo**                             | Combined indicator (weather + curvature + lighting) captures compound effects very effectively. |
| 3    | **Num reported accidents**                 | Areas with prior incidents remain risky — strong spatial correlation.                           |
| 4    | **Lane-speed factor / Speed limit**        | High speeds on narrow or curved roads increase risk nonlinearly.                                |
| 5    | **Weather risk / Visibility**              | Rain, fog, and poor lighting amplify hazard likelihood.                                         |
| 6    | **Lighting conditions (night, dim)**       | Nighttime or poor lighting conditions strongly influence outcomes.                              |
| 7    | **Public road / Holiday / School season**  | Contextual factors (traffic volume, pedestrian density).                                        |
| 8    | **Temporal factors (sin_time, rush_hour)** | Captures time-of-day cyclic risk patterns.                                                      |

✅ These top features explain **≈90% of variance** in predictions — your domain-engineered variables like  
`risk_combo`, `visibility_risk`, and `lane_speed_factor` are high-impact and realistic.

---

## 🧠 3. Recommendations for Further Improvement

### 1. Cross-validation robustness
* Use **StratifiedKFold (on risk bins)** to ensure stable generalization.  
* Try **repeated KFold** for smoother average estimates.

### 2. Advanced feature engineering
* Compute **interaction terms**, e.g.:
  - `curvature × speed_limit`
  - `lighting × weather_risk`
* Use **historical accident density smoothing** instead of raw counts (kernel density estimate).

### 3. Model ensembling
* Weighted average: `0.6 * LGBM + 0.4 * XGB`
* Stacking (meta-model on top of XGB + LGBM predictions)

### 4. Model calibration
* Apply **Isotonic Regression** or **Platt scaling** to ensure probabilistic calibration  
  (useful for ranking risk).

### 5. SHAP analysis
* For explainability, compute `shap.summary_plot()` for LGBM/XGB → identifies nonlinear feature effects.

---

## 🔍 Next Step Options

Would you like me to:

1. 📈 **Build and show an ensemble model** (LGBM + XGB) performance, or  
2. 🔍 **Perform SHAP-based interpretability** to visualize top features and how they affect risk prediction?


In [ ]:
lgbm_model

In [ ]:
xgb_model

In [ ]:
import pickle
MODEL_PATH = r"../data/road_accidents/xgb_model.pkl"
with open(MODEL_PATH, "wb") as f:
    pickle.dump(xgb_model, f)

MODEL_PATH = r"../data/road_accidents/lgbm_model.pkl"
with open(MODEL_PATH, "wb") as f:
    pickle.dump(lgbm_model, f)

In [ ]:
models_model_df = pd.DataFrame()
models_model_df['xgb_pred'] = xgb_model.predict(X_train)
models_model_df['lgbm_pred'] = lgbm_model.predict(X_train)
models_model_df["accident_risk"] = y_train


In [ ]:
stack_lgbm_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
X_train_stack = models_model_df[['xgb_pred', 'lgbm_pred']]
stack_lgbm_model.fit(X_train_stack, y_train)
xgb_model.predict(X_test)
lgbm_model.predict(X_test)
X_test_stack = pd.DataFrame()
X_test_stack['xgb_pred'] = xgb_model.predict(X_test)
X_test_stack['lgbm_pred'] = lgbm_model.predict(X_test)
y_pred_stack_lgbm = stack_lgbm_model.predict(X_test_stack)

print('RMSE (LGBM):', mean_squared_error(y_test, y_pred_stack_lgbm))
print('R2 Score (LGBM):', r2_score(y_test, y_pred_stack_lgbm))

X_test_stack["new_pred"] = X_test_stack['xgb_pred'] * 0.5   + X_test_stack['lgbm_pred'] * 0.5

print('RMSE (LGBM):', mean_squared_error(y_test, X_test_stack["new_pred"]))
print('R2 Score (LGBM):', r2_score(y_test, X_test_stack["new_pred"]))


In [ ]:
RMSE (LGBM): 0.0031975138667271804
R2 Score (LGBM): 0.8841992115478337
RMSE (LGBM): 0.0031753077157787996
R2 Score (LGBM): 0.8850034269149878

In [165]:
(0.0031753077157787996)**(1/2)

0.05634986881776034

# EDA

In [ ]:
def get_input(data):
    if not data:
        df = pd.read_csv(r"../data/road_accidents/train.csv")
    else:
        df = pd.read_csv(data)

    return df

In [ ]:
df.head()

In [ ]:
df.lighting.value_counts()

In [ ]:
df[(df.lighting == "daylight") & (df.weather == "foggy")].sort_values(by=["curvature", "lighting", 'weather', "accident_risk"], ascending=False).head(50)

In [ ]:
def weather_lighting_interactions(df):
    df["rainy_night"] = ((df["weather"] == "rainy") & (df["lighting"] == "night")).astype(int)
    df["rainy_daylight"] = ((df["weather"] == "rainy") & (df["lighting"] == "daylight")).astype(int)
    df["rainy_dim"] = ((df["weather"] == "rainy") & (df["lighting"] == "dim")).astype(int)
    df["foggy_night"] = ((df["weather"] == "foggy") & (df["lighting"] == "night")).astype(int)
    df["foggy_daylight"] = ((df["weather"] == "foggy") & (df["lighting"] == "daylight")).astype(int)
    df["foggy_dim"] = ((df["weather"] == "foggy") & (df["lighting"] == "dim")).astype(int)
    df["clear_dim"] = ((df["weather"] == "clear") & (df["lighting"] == "dim")).astype(int)
    df["clear_night"] = ((df["weather"] == "clear") & (df["lighting"] == "night")).astype(int)
    df["clear_daylight"] = ((df["weather"] == "clear") & (df["lighting"] == "daylight")).astype(int)
    return df 


In [ ]:
# | Feature                                                     | Description                                            | Rationale |
# | ----------------------------------------------------------- | ------------------------------------------------------ | --------- |
# | `curvature * speed_limit`                                   | Curves + speed → dangerous at high speed               |           |
# | `curvature * num_lanes`                                     | Wide roads offset curve danger                         |           |
# | `speed_limit * num_reported_accidents`                      | Indicates historical risk exposure                     |           |
# | `speed_limit * school_season`                               | Higher speed limits during school time = higher risk   |           |
# | `curvature * road_type_highway`                             | Curves on highways are more severe than urban curves   |           |
# | `speed_limit * road_type_rural`                             | High-speed rural roads riskier due to lower visibility |           |
# | `curvature * weather_risk` *(optional if you have weather)* | Bad weather amplifies curvature effects                |           |


# === Create domain-driven interaction features ===
def create_interaction_features(df):
    df = pd.get_dummies(df, columns=['road_type'], drop_first=False)

    # === Create Domain Interaction Features ===
    df['curvature_speed'] = df['curvature'] * df['speed_limit']
    df['curvature_lanes'] = df['curvature'] * df['num_lanes']
    df['speed_prev_acc'] = df['speed_limit'] * df['num_reported_accidents']
    df['speed_school']   = df['speed_limit'] * df['school_season']
    df['curv_highway']   = df['curvature'] * df['road_type_highway']
    df['speed_rural']    = df['speed_limit'] * df['road_type_rural']
    return df

In [ ]:
df.lighting.value_counts(), df.weather.value_counts()

In [ ]:
#        'road_type', 'lighting',
#        'weather', 'road_signs_present', 'public_road', 'time_of_day',
#        'holiday', 'school_season', 'num_reported_accidents', 'accident_risk',
#        'rainy_night', 'rainy_dim', 'foggy_night', 'rainy_daylight',
#        'foggy_daylight', 'foggy_dim', 'clear_dim', 'clear_night',
#        'clear_daylight'
from sklearn.preprocessing import OneHotEncoder, LabelEncoder


def ohe_encoding(df, cat_cols):
    
    encoder = OneHotEncoder(sparse_output=False, drop='first')
    encoded_cat = encoder.fit_transform(df[cat_cols])
    encoded_cat_df = pd.DataFrame(encoded_cat, columns=encoder.get_feature_names_out(cat_cols))
    df.reset_index(drop=True, inplace=True)
    df = pd.concat([df, encoded_cat_df], axis=1)
    df.drop(columns=cat_cols, inplace=True)
    return df

def le_encoding(df, cat_cols):
    label_encoder = LabelEncoder()
    for i in cat_cols:
        df[i] = label_encoder.fit_transform(df[i])
    return df


In [ ]:
from sklearn.model_selection import train_test_split
def split_data(df):
    X = df.drop(columns=['accident_risk', 'id'])
    y = df['accident_risk']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    return X_train, X_test, y_train, y_test


In [171]:
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score, accuracy_score, root_mean_squared_error

def lgbm_model_training(split_data):
    X_train, X_test, y_train, y_test = split_data
    lgbm_model = LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    lgbm_model.fit(X_train, y_train)
    y_pred_lgbm = lgbm_model.predict(X_test)
    print('RMSE (LGBM):', root_mean_squared_error(y_test, y_pred_lgbm))
    print('R2 Score (LGBM):', r2_score(y_test, y_pred_lgbm))
    print('Accuracy Score (LGBM):', accuracy_score(y_test.round(), y_pred_lgbm.round()))
    return lgbm_model

def xgb_model_training(split_data):
    X_train, X_test, y_train, y_test = split_data
    xgb_model = XGBRegressor(
        n_estimators=300,       # number of trees
        learning_rate=0.05,      # step size shrinkage
        max_depth=6,            # depth of each tree
        subsample=0.8,          # fraction of samples used per tree
        colsample_bytree=0.8,   # fraction of features used per tree
        random_state=42,
        n_jobs=-1,
        tree_method='hist',      # efficient tree method
        objective='reg:squarederror'  # regression task
    )
    xgb_model.fit(X_train, y_train)
    y_pred = xgb_model.predict(X_test)
    print('RMSE (XGB):', root_mean_squared_error(y_test, y_pred))
    print('R2 Score (XGB):', r2_score(y_test, y_pred))
    print('Accuracy Score (XGB):', accuracy_score(y_test.round(), y_pred.round()))
    return xgb_model

def cat_model_training(split_data, cat_features):
    X_train, X_test, y_train, y_test = split_data
    cat_model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        subsample=0.8,
        random_seed=42,
        verbose=False
    )
    cat_model.fit(X_train, y_train, cat_features=cat_features)
    y_pred_cat = cat_model.predict(X_test)
    print('RMSE (CatBoost):', root_mean_squared_error(y_test, y_pred_cat))
    print('R2 Score (CatBoost):', r2_score(y_test, y_pred_cat))
    print('Accuracy Score (CatBoost):', accuracy_score(y_test.round(), y_pred_cat.round()))
    return cat_model


In [169]:
df = get_input(None)

cat_features = ['road_type', 'lighting', 'weather', 'time_of_day', 'road_signs_present', 'public_road', 'holiday', 'school_season']

cat_splits = split_data(df.copy())

cat_model_training(cat_splits, cat_features)

RMSE (CatBoost): 0.0032024241713374383
R2 Score (CatBoost): 0.8840213805300154
Accuracy Score (CatBoost): 0.937702195053645


In [170]:
0.0032024241713374383**0.5

0.05658996528835689

In [160]:
cat_splits[0].head()

,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,road_type_highway,road_type_rural,road_type_urban,curvature_speed,curvature_lanes,speed_prev_acc,speed_school,curv_highway,speed_rural
143159,2,0.43,60,dim,clear,False,False,afternoon,True,True,2,False,True,False,25.80,0.86,120,60,0.00,60
20172,4,0.18,25,night,clear,False,False,afternoon,True,False,1,True,False,False,4.50,0.72,25,0,0.18,0
57926,1,0.20,70,night,clear,False,True,morning,True,True,1,False,False,True,14.00,0.20,70,70,0.00,0
193319,3,0.81,60,daylight,foggy,True,False,evening,True,True,1,False,False,True,48.60,2.43,60,60,0.00,0
213938,3,0.43,35,night,rainy,False,True,morning,True,True,2,False,True,False,15.05,1.29,70,35,0.00,35


In [174]:
# Runner 
df_raw = get_input(None)
# df = weather_lighting_interactions(df)
df = create_interaction_features(df_raw.copy())
num_cols = ['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents', 'accident_risk']
cat_cols = ['road_signs_present', 'public_road', 'time_of_day']
bool_cols = ['holiday', 'school_season']
cols_to_drop = ['lighting', 'weather']

cat_cols.extend(cols_to_drop)
# df.drop(columns=cols_to_drop, inplace=True)
df[bool_cols] = df[bool_cols].astype(int)

df_ohe = ohe_encoding(df.copy(), cat_cols)
df_label = le_encoding(df.copy(), cat_cols)

label_split = split_data(df_label.copy())
ohe_split = split_data(df_ohe.copy())
cat_splits = split_data(df_raw.copy())

print("LabelEncoder")
le_lgbm = lgbm_model_training(label_split)


print("-------------------")
le_xgb = xgb_model_training(label_split)

print("-------------------")
print("OHEEncoder")
ohe_lgbm = lgbm_model_training(ohe_split)

print("-------------------")
ohe_xgb = xgb_model_training(ohe_split)
    
print("-------------------")
cat_features = ['road_type', 'lighting', 'weather', 'time_of_day', 'road_signs_present', 'public_road', 'holiday', 'school_season']
cat_model = cat_model_training(cat_splits, cat_features=cat_features)


LabelEncoder
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008784 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 805
[LightGBM] [Info] Number of data points in the train set: 414203, number of used features: 20
[LightGBM] [Info] Start training from score 0.352605
RMSE (LGBM): 0.0563996277043461
R2 Score (LGBM): 0.884800245329479
Accuracy Score (LGBM): 0.9388803584707053
-------------------
RMSE (XGB): 0.05627718971944744
R2 Score (XGB): 0.8852998768451056
Accuracy Score (XGB): 0.9393728694073452
-------------------
OHEEncoder
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007856 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 808
[LightGBM] [Info] Number of data points in the train s

In [ ]:
le_lgbm.feature_importances_

In [ ]:
importances = pd.Series(le_xgb.feature_importances_ * 100, index=label_split[0].columns)
low_features = importances[importances < 10].index
# X.drop(columns=low_features, inplace=True)


In [ ]:
importances

In [ ]:
ohe_split[0].columns

In [ ]:
df.columns

In [ ]:
df

In [179]:
test = pd.read_csv(r"../data/road_accidents/test.csv")

In [ ]:
# Runner 
# df = get_input(r"../data/road_accidents/test.csv")
df = test.copy()
# df = weather_lighting_interactions(df)
df = create_interaction_features(df.copy())
num_cols = ['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents', 'accident_risk']
cat_cols = ['road_signs_present', 'public_road', 'time_of_day']
bool_cols = ['holiday', 'school_season']
cols_to_drop = ['lighting', 'weather']

cat_cols.extend(cols_to_drop)
# df.drop(columns=cols_to_drop, inplace=True)
df[bool_cols] = df[bool_cols].astype(int)

# df_ohe = ohe_encoding(df.copy(), cat_cols)
df_label = le_encoding(df.copy(), cat_cols)

# label_split = split_data(df_label.copy())
# ohe_split = split_data(df_ohe.copy())
df_id = df_label['id']
df_label.drop(columns=['id'], inplace=True)
cat.predict(df_label)
preds = le_lgbm.predict(df_label)
submission = pd.DataFrame({'id': df_id, 'accident_risk': preds})




In [177]:
submission.to_csv(r"../data/road_accidents/submission_xgb.csv", index=False)

In [ ]:
from sklearn.metrics import log_loss

preds = le_lgbm.predict(label_split[1])
# ans_df = pd.DataFrame({'accident_risk': preds})
X_test = label_split[1]
y_test = label_split[3]
log_loss(pd.DataFrame({'accident_risk': preds})['accident_risk'], y_test)
# print(loss)


In [181]:
test

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents
0,517754,highway,2,0.34,45,night,clear,True,True,afternoon,True,True,1
1,517755,urban,3,0.04,45,dim,foggy,True,False,afternoon,True,False,0
2,517756,urban,2,0.59,35,dim,clear,True,False,afternoon,True,True,1
3,517757,rural,4,0.95,35,daylight,rainy,False,False,afternoon,False,False,2
4,517758,highway,2,0.86,35,daylight,clear,True,False,evening,False,True,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
172580,690334,rural,2,0.01,45,dim,rainy,False,False,afternoon,True,True,2
172581,690335,rural,1,0.74,70,daylight,foggy,False,True,afternoon,False,False,2
172582,690336,urban,2,0.14,70,dim,clear,False,False,evening,True,True,1
172583,690337,urban,1,0.09,45,daylight,foggy,True,True,morning,False,True,0


In [190]:
# Runner 
# df = get_input(r"../data/road_accidents/test.csv")
df = test.copy()
 
df_id = df['id']
df.drop(columns=['id'], inplace=True)
preds = cat_model.predict(df)
submission = pd.DataFrame({'id': df_id, 'accident_risk': preds})

In [191]:
submission.to_csv(r"../data/road_accidents/submission_catboost.csv", index=False)